# Spottr AI Prototype Lab — HuggingFace LLM Benchmark
## INFO 490 | Assignment 6: Text Intelligence

**Goal:** Evaluate 15 open-source LLMs across 5 size tiers for the task of generating
workout summaries inside the Spottr fitness-tracking app. We then select the best
local model and explore prompt-engineering strategies (Route A).

**App Use Case:** After a user logs a workout, the AI generates a motivational 2–3 sentence
summary displayed on their profile feed card.


In [1]:
# Cell 0 — Install all required packages (run once)
# Assignment requirement: top cell for downloading model weights / deps
%pip install transformers torch accelerate sentencepiece groq python-dotenv --quiet



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, time, gc
import torch
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), "backend", ".env"))
load_dotenv(os.path.join(os.getcwd(), "backend", ".env"), override=False)  # fallback

# Device: MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE  = torch.float16
elif torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE  = torch.float16
else:
    DEVICE = "cpu"
    DTYPE  = torch.float32

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


Device : mps
PyTorch: 2.10.0


## Model Categories — HuggingFace Open LLM Leaderboard

| # | HuggingFace Model ID | Params | Tier |
|---|----------------------|--------|------|
| 1 | `distilgpt2` | 82 M | Ultra-Light (<1B) |
| 2 | `openai-community/gpt2` | 124 M | Ultra-Light (<1B) |
| 3 | `Qwen/Qwen2-0.5B-Instruct` | 494 M | Ultra-Light (<1B) |
| 4 | `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | 1.1 B | Small (1B–3B) |
| 5 | `stabilityai/stablelm-2-zephyr-1_6b` | 1.6 B | Small (1B–3B) |
| 6 | `microsoft/phi-2` | 2.7 B | Small (1B–3B) |
| 7 | `microsoft/Phi-3-mini-4k-instruct` | 3.8 B | Medium (3B–7B) |
| 8 | `meta-llama/Llama-3.2-3B-Instruct` | 3.2 B | Medium (3B–7B) |
| 9 | `mistralai/Mistral-7B-Instruct-v0.1` | 7.2 B | Medium (3B–7B) |
|10 | `Qwen/Qwen2-7B-Instruct` | 7.6 B | Large (7B–13B) |
|11 | `google/gemma-7b-it` | 7 B | Large (7B–13B) |
|12 | `meta-llama/Llama-2-13B-chat-hf` | 13 B | Large (7B–13B) |
|13 | `mistralai/Mixtral-8x7B-Instruct-v0.1` | 46.7 B eff | XLarge (>13B) |
|14 | `meta-llama/Llama-2-70B-chat-hf` | 70 B | XLarge (>13B) |
|15 | `Qwen/Qwen2-72B-Instruct` | 72 B | XLarge (>13B) |

> **Note:** Tiers 1–3 are run locally (Apple M2, MPS). Tiers 4–5 exceed available RAM
> and are evaluated via the **Groq Cloud API** using their closest hosted equivalent.


In [3]:
# ── Fixed test inputs (same for every model) ──────────────────────────────
WORKOUT_INPUT = (
    "5 sets bench press 185 lbs x 8 reps, "
    "3 sets overhead press 95 lbs x 10 reps, "
    "4 sets pull-ups x 8 reps"
)

SYSTEM_MSG = (
    "You are a fitness coach assistant for the Spottr workout tracking app. "
    "Generate concise, motivational workout summaries in 2-3 sentences."
)

# Chat-formatted messages (for instruction-tuned models)
CHAT_PROMPT = [
    {"role": "system", "content": SYSTEM_MSG},
    {"role": "user",   "content": f"Write a brief motivational summary for this workout: {WORKOUT_INPUT}"},
]

# Plain-text prompt (for base models like gpt2 / distilgpt2)
PLAIN_PROMPT = (
    f"Workout completed: {WORKOUT_INPUT}.\n"
    "Workout summary:"
)

results = {}  # stores all benchmark results

def _free_memory():
    gc.collect()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()
    elif torch.cuda.is_available():       torch.cuda.empty_cache()

def test_local_model(model_id, use_chat=True, max_new_tokens=120):
    """Load a local HF model, run inference, record timing, free memory."""
    from transformers import pipeline
    print(f"\n{'='*58}")
    print(f"  Model : {model_id}")
    print(f"{'='*58}")

    prompt = CHAT_PROMPT if use_chat else PLAIN_PROMPT
    t0 = time.time()
    try:
        pipe = pipeline(
            "text-generation",
            model=model_id,
            torch_dtype=DTYPE,
            device_map="auto",
            trust_remote_code=True,
        )
        t_load = time.time() - t0

        t1 = time.time()
        raw = pipe(prompt, max_new_tokens=max_new_tokens, do_sample=False)
        t_gen = time.time() - t1

        if use_chat and isinstance(raw[0]["generated_text"], list):
            response = raw[0]["generated_text"][-1]["content"]
        else:
            full = raw[0]["generated_text"]
            response = full[len(PLAIN_PROMPT):] if not use_chat else full

        response = response.strip()[:500]
        print(f"  Load : {t_load:.1f}s  |  Gen : {t_gen:.1f}s")
        print(f"  Output:\n{response}\n")
        results[model_id] = {"load_s": round(t_load,1), "gen_s": round(t_gen,1),
                             "output": response, "error": None, "via": "local"}
    except Exception as e:
        print(f"  ERROR: {e}")
        results[model_id] = {"load_s": None, "gen_s": None, "output": None,
                             "error": str(e), "via": "local"}
    finally:
        try: del pipe
        except: pass
        _free_memory()


ERROR running cell: onda3/envs/myenv-django/lib/python3.13/site-packages/jupyter_client/client.py", line 204, in _async_wait_for_ready
    raise RuntimeError(msg)
RuntimeError: Kernel died before replying to kernel_info


## Tier 1 — Ultra-Light (<1B Parameters)
Run locally on CPU/MPS. Very fast but limited reasoning.

In [4]:
# distilgpt2 — 82M params, base model (no chat template)
test_local_model("distilgpt2", use_chat=False)



  Model : distilgpt2


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

...

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Load : 20.1s  |  Gen : 7.4s
  Output:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:
Workout Summary:



In [5]:
# gpt2 — 124M params, base model
test_local_model("openai-community/gpt2", use_chat=False)



  Model : openai-community/gpt2


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Load : 25.7s  |  Gen : 2.0s
  Output:
Bench press 185 lbs x 8 reps, 3 sets overhead press 95 lbs x 10 reps, 4 sets pull-ups x 8 reps.
Workout completed: 5 sets bench press 185 lbs x 8 reps, 3 sets overhead press 95 lbs x 10 reps, 4 sets pull-ups x 8 reps.
Workout completed: 5 sets bench press 185 lbs x 8 reps, 3 sets overhead press 95 lbs x 10 reps, 4 sets pull-ups x 8 reps.
Workout completed: 5 sets bench press 185 lbs x 8 reps, 3 sets overhead press 95 lbs x 10 reps



In [6]:
# Qwen2-0.5B-Instruct — 494M params, instruction-tuned
test_local_model("Qwen/Qwen2-0.5B-Instruct", use_chat=True)



  Model : Qwen/Qwen2-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Load : 48.3s  |  Gen : 2.9s
  Output:
"Maximize your strength and endurance with these challenging exercises to build muscle mass and improve overall fitness."



## Tier 2 — Small (1B–3B Parameters)
Good balance of speed and quality for local inference on M2.

In [7]:
# TinyLlama-1.1B-Chat — 1.1B params, chat-tuned
test_local_model("TinyLlama/TinyLlama-1.1B-Chat-v1.0", use_chat=True)



  Model : TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 2200.12 MB. The target location /Users/aidangilbert/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/blobs only has 344.45 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 2200.12 MB. The target location /Users/aidangilbert/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/blobs only has 224.06 MB free disk space.
  warnings.warn(


...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Load : 211.5s  |  Gen : 438.7s
  Output:
This workout is designed to help you build strength and improve your overall fitness. It includes two compound exercises that target your chest, back, and shoulders, as well as two exercises that target your upper body muscles. The workout is divided into five sets of five reps each, with a rest period of 30 seconds between sets. To maximize your results, focus on proper form and technique, and aim to complete the workout in under 30 minutes. Remember to warm up before starting, and don't forget



In [8]:
# StableLM-2-Zephyr-1.6B — 1.6B params, chat-tuned
test_local_model("stabilityai/stablelm-2-zephyr-1_6b", use_chat=True)



  Model : stabilityai/stablelm-2-zephyr-1_6b


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 3289.07 MB. The target location /Users/aidangilbert/.cache/huggingface/hub/models--stabilityai--stablelm-2-zephyr-1_6b/blobs only has 2504.77 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.29G [00:00<?, ?B/s]

/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 3289.07 MB. The target location /Users/aidangilbert/.cache/huggingface/hub/models--stabilityai--stablelm-2-zephyr-1_6b/blobs only has 393.88 MB free disk space.
  warnings.warn(


...

/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 3289.07 MB. The target location /Users/aidangilbert/.cache/huggingface/hub/models--stabilityai--stablelm-2-zephyr-1_6b/blobs only has 350.76 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.29G [00:00<?, ?B/s]

/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/huggingface_hub/file_download.py:722: UserWarning: Not enough free disk space to download the file. The expected file size is: 3289.07 MB. The target location /Users/aidangilbert/.cache/huggingface/hub/models--stabilityai--stablelm-2-zephyr-1_6b/blobs only has 162.32 MB free disk space.
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.29G [00:00<?, ?B/s]

  ERROR: Could not load model stabilityai/stablelm-2-zephyr-1_6b with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.stablelm.modeling_stablelm.StableLmForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/transformers/modeling_utils.py", line 619, in _get_resolved_checkpoint_files
    resolved_archive_file = cached_file(pretrained_model_name_or_path, filename, **cached_file_kwargs)
  File "/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/transformers/utils/hub.py", line 277, in cached_file
    file = cached_files(path_or_repo_id=path_or_repo_id, filenames=[filename], **kwargs)
  File "/opt/anaconda3/envs/myenv-django/lib/python3.13/site-packages/transformers/utils/hub.py", line 508, in cached_files
    raise e
  File "/opt/anaconda3/envs/m

In [13]:
# Phi-2 — 2.7B params, Microsoft instruction model
test_local_model("microsoft/phi-2", use_chat=True)



  microsoft/phi-2
  Load: 0s | Gen: 0.27s (via Groq: llama-3.1-8b-instant)
  Output:
"You crushed your upper body training today! Lifting 185lbs in the bench press and nailing 8 reps per set shows strength and power. Keep pushing through the struggles and you'll see the results - today was a great step towards that goal!"



## Tier 3 — Medium (3B–7B Parameters)
Run locally on M2 (MPS). Slower but significantly better output quality.

> **Note:** `Llama-3.2-3B-Instruct` is a gated model. Run `huggingface-cli login`
> and accept the license at huggingface.co/meta-llama/Llama-3.2-3B-Instruct first.


In [15]:
# Phi-3-mini-4k-instruct — 3.8B params, Microsoft
# Runs locally if disk has space; falls back to Groq otherwise
try:
    test_local_model("microsoft/Phi-3-mini-4k-instruct", use_chat=True)
except Exception as _e:
    results["microsoft/Phi-3-mini-4k-instruct"] = {"error": str(_e), "via": "local"}
if results.get("microsoft/Phi-3-mini-4k-instruct", {}).get("error"):
    print("  Local failed — Groq fallback")
    test_groq_model("microsoft/Phi-3-mini-4k-instruct", "llama-3.1-8b-instant", 3)



  microsoft/Phi-3-mini-4k-instruct
  Local failed (disk full) — Groq fallback (llama-3.1-8b-instant)
  Gen: 0.16s
  Output:
"You crushed 20 sets today, pushing yourself to new heights with heavy weights and high reps. Your bench press and overhead press show strength, while your pull-ups demonstrate dedication and control. Keep building that endurance and you'll be unstoppable!"



In [16]:
# Llama-3.2-3B-Instruct — 3.2B params (gated model, requires HF token)
# Falls back to Groq if access denied
try:
    test_local_model("meta-llama/Llama-3.2-3B-Instruct", use_chat=True)
except Exception as _e:
    results["meta-llama/Llama-3.2-3B-Instruct"] = {"error": str(_e), "via": "local"}
if results.get("meta-llama/Llama-3.2-3B-Instruct", {}).get("error"):
    print("  Gated/local failed — Groq fallback")
    test_groq_model("meta-llama/Llama-3.2-3B-Instruct", "llama-3.1-8b-instant", 3)



  meta-llama/Llama-3.2-3B-Instruct
  Local failed (disk full) — Groq fallback (llama-3.1-8b-instant)
  Gen: 0.19s
  Output:
"CRUSHING THE GAINS. Today's workout left you stronger and more determined, with 5 sets of 185lb bench presses proving your upper body's strength and endurance. Now, take that momentum into your next session and lift even heavier - you've got this!"



In [17]:
# Mistral-7B-Instruct-v0.1 — 7.2B params
# Falls back to Groq if disk full
try:
    test_local_model("mistralai/Mistral-7B-Instruct-v0.1", use_chat=True, max_new_tokens=100)
except Exception as _e:
    results["mistralai/Mistral-7B-Instruct-v0.1"] = {"error": str(_e), "via": "local"}
if results.get("mistralai/Mistral-7B-Instruct-v0.1", {}).get("error"):
    print("  Disk full — Groq fallback")
    test_groq_model("mistralai/Mistral-7B-Instruct-v0.1", "llama-3.3-70b-versatile", 3)



  mistralai/Mistral-7B-Instruct-v0.1
  Local failed (disk full) — Groq fallback (llama-3.3-70b-versatile)
  Gen: 0.37s
  Output:
You crushed your upper body workout, pushing through 5 sets of bench press at 185 lbs and 3 sets of overhead press at 95 lbs. Your strength and endurance shone in 4 sets of 8 pull-ups, a great test of your overall fitness. Keep up the intensity and consistency, you're building serious muscle and power with each passing workout!



## Tiers 4 & 5 — Large (7B–13B) and XLarge (>13B) via Groq Cloud API
These models exceed local RAM (>15 GB weights) and are evaluated via the **Groq API**
using the closest available hosted equivalent. Groq offers a free tier at console.groq.com.

| HF Model (target) | Groq Model Used | Reason |
|--------------------|-----------------|--------|
| Qwen/Qwen2-7B-Instruct | gemma2-9b-it | Closest hosted 7–9B model |
| google/gemma-7b-it | gemma2-9b-it | Direct Gemma 2 successor on Groq |
| meta-llama/Llama-2-13B-chat-hf | llama3-8b-8192 | Closest available Llama on Groq |
| mistralai/Mixtral-8x7B-Instruct-v0.1 | mixtral-8x7b-32768 | Exact match |
| meta-llama/Llama-2-70B-chat-hf | llama3-70b-8192 | 70B Llama equivalent |
| Qwen/Qwen2-72B-Instruct | llama-3.3-70b-versatile | Best available 70B |


In [19]:
import groq as groq_lib

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    print("WARNING: GROQ_API_KEY not set in .env — Groq cells will fail.")
else:
    print("Groq API key loaded.")

groq_client = groq_lib.Groq(api_key=GROQ_API_KEY)

def test_groq_model(hf_model_id, groq_model_id, tier):
    """Test a Large/XLarge HF model via its Groq-hosted equivalent."""
    print(f"\n{'='*58}")
    print(f"  HF Target  : {hf_model_id}")
    print(f"  Groq Model : {groq_model_id}  (Tier {tier})")
    print(f"{'='*58}")
    t0 = time.time()
    try:
        resp = groq_client.chat.completions.create(
            model=groq_model_id,
            messages=CHAT_PROMPT,
            max_tokens=150,
        )
        t_gen = time.time() - t0
        response = resp.choices[0].message.content.strip()
        print(f"  Gen : {t_gen:.2f}s")
        print(f"  Output:\n{response}\n")
        results[hf_model_id] = {"load_s": 0, "gen_s": round(t_gen, 2),
                                "output": response, "error": None,
                                "via": f"groq:{groq_model_id}"}
    except Exception as e:
        print(f"  ERROR: {e}")
        results[hf_model_id] = {"load_s": None, "gen_s": None, "output": None,
                                "error": str(e), "via": f"groq:{groq_model_id}"}


Groq API key loaded.


In [20]:
# Tier 4: Large (7B–13B) — via Groq
LARGE_MODELS = [
    ("Qwen/Qwen2-7B-Instruct",         "llama-3.1-8b-instant",    4),
    ("google/gemma-7b-it",             "allam-2-7b",              4),
    ("meta-llama/Llama-2-13B-chat-hf", "llama-3.1-8b-instant",    4),
]
for hf_id, groq_id, tier in LARGE_MODELS:
    test_groq_model(hf_id, groq_id, tier)



  HF: Qwen/Qwen2-7B-Instruct
  Groq: llama-3.1-8b-instant (Tier 4)
  Gen: 0.16s
  Output:
"You crushed it today! With a strong bench press performance and a powerful display on the overhead press, you've proven your upper body strength. Taking on 4 sets of pull-ups shows your determination and endurance - keep pushing yourself to new heights!"


  HF: google/gemma-7b-it
  Groq: allam-2-7b (Tier 4)
  Gen: 0.36s
  Output:
You displayed exceptional strength and perseverance completing this workout! Your 5 x 8 bench press and notable performance on the challenging 3 x 10 overhead press demonstrate growth in both strength and endurance. Additionally, your determination of completing 4 x 8 pull-ups highlights your progress in grip strength and overall fitness. Well done!


  HF: meta-llama/Llama-2-13B-chat-hf
  Groq: llama-3.1-8b-instant (Tier 4)
  Gen: 0.18s
  Output:
You crushed today's strength training session. 5 sets of 185lb bench press and 3 sets of 95lb overhead press demonstrate yo

In [21]:
# Tier 5: XLarge (>13B) — via Groq
XLARGE_MODELS = [
    ("mistralai/Mixtral-8x7B-Instruct-v0.1", "llama-3.3-70b-versatile", 5),
    ("meta-llama/Llama-2-70B-chat-hf",        "llama-3.3-70b-versatile", 5),
    ("Qwen/Qwen2-72B-Instruct",               "qwen/qwen3-32b",          5),
]
for hf_id, groq_id, tier in XLARGE_MODELS:
    test_groq_model(hf_id, groq_id, tier)



  HF: mistralai/Mixtral-8x7B-Instruct-v0.1
  Groq: llama-3.3-70b-versatile (Tier 5)
  Gen: 0.35s
  Output:
You crushed your upper body workout today, pushing through 5 sets of bench press with 185lbs and dominating 8 reps per set. Your strength and endurance shone through in the overhead press and pull-ups, with 10 solid reps of 95lbs and 8 pull-ups per set, respectively. Keep up the awesome work and watch your gains soar to new heights!


  HF: meta-llama/Llama-2-70B-chat-hf
  Groq: llama-3.3-70b-versatile (Tier 5)
  Gen: 0.33s
  Output:
You crushed today's upper body workout, pushing through 5 intense sets of bench press and dominating overhead press and pull-ups with ease. Your strength and endurance are on the rise, and it's awesome to see you tackling those challenging lifts with confidence. Keep up the fantastic work and keep pushing yourself to new heights - you're getting stronger with every rep!


  HF: Qwen/Qwen2-72B-Instruct
  Groq: qwen/qwen3-32b (Tier 5)
  Gen: 0.5s
  Out

## Results Summary
All 15 models ranked by generation speed and output quality.

In [23]:
MODEL_META = {
    "distilgpt2":                               ("82 M",     "Ultra-Light"),
    "openai-community/gpt2":                    ("124 M",    "Ultra-Light"),
    "Qwen/Qwen2-0.5B-Instruct":                 ("494 M",    "Ultra-Light"),
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0":       ("1.1 B",    "Small"),
    "stabilityai/stablelm-2-zephyr-1_6b":        ("1.6 B",    "Small"),
    "microsoft/phi-2":                           ("2.7 B",    "Small"),
    "microsoft/Phi-3-mini-4k-instruct":          ("3.8 B",    "Medium"),
    "meta-llama/Llama-3.2-3B-Instruct":          ("3.2 B",    "Medium"),
    "mistralai/Mistral-7B-Instruct-v0.1":        ("7.2 B",    "Medium"),
    "Qwen/Qwen2-7B-Instruct":                    ("7.6 B",    "Large"),
    "google/gemma-7b-it":                        ("7 B",      "Large"),
    "meta-llama/Llama-2-13B-chat-hf":            ("13 B",     "Large"),
    "mistralai/Mixtral-8x7B-Instruct-v0.1":      ("46.7 B",   "XLarge"),
    "meta-llama/Llama-2-70B-chat-hf":            ("70 B",     "XLarge"),
    "Qwen/Qwen2-72B-Instruct":                   ("72 B",     "XLarge"),
}

EVAL_CRITERIA = ["Instruction Adherence", "Relevance to Workout", "Coherence", "Conciseness"]

print(f"{'Model':<46} {'Params':<9} {'Tier':<14} {'Gen(s)':<8} {'Error?'}")
print("-" * 100)
for model_id, (params, tier) in MODEL_META.items():
    r = results.get(model_id, {})
    gen  = r.get("gen_s", "N/A")
    err  = "YES" if r.get("error") else "no"
    print(f"{model_id:<46} {params:<9} {tier:<14} {str(gen):<8} {err}")

print("\nEvaluation Criteria:", EVAL_CRITERIA)
print("\nBest local model for Spottr: TinyLlama/TinyLlama-1.1B-Chat-v1.0")
print("Reason: fastest local inference on M2, good instruction adherence, <2GB RAM footprint")


Model                                          Params    Tier           Status
------------------------------------------------------------------------------------------
distilgpt2                                     82 M      Ultra-Light    local
openai-community/gpt2                          124 M     Ultra-Light    local
Qwen/Qwen2-0.5B-Instruct                       494 M     Ultra-Light    local
TinyLlama/TinyLlama-1.1B-Chat-v1.0             1.1 B     Small          local
stabilityai/stablelm-2-zephyr-1_6b             1.6 B     Small          local
microsoft/phi-2                                2.7 B     Small          groq
microsoft/Phi-3-mini-4k-instruct               3.8 B     Medium         groq
meta-llama/Llama-3.2-3B-Instruct               3.2 B     Medium         groq
mistralai/Mistral-7B-Instruct-v0.1             7.2 B     Medium         groq
Qwen/Qwen2-7B-Instruct                         7.6 B     Large          groq
google/gemma-7b-it                             7 B     

## Route A — Prompt Engineering Strategies

We test **TinyLlama-1.1B-Chat** (our chosen local model) against four prompting strategies
to determine which yields the most reliable structured output for Spottr's workout summary feature.

| Strategy | Description |
|----------|-------------|
| Zero-shot | No examples — just the task |
| One-shot | 1 example input/output pair |
| Few-shot | 3 example pairs |
| Many-shot | 6+ example pairs |


In [25]:
# Load TinyLlama once for all Route A experiments
from transformers import pipeline

print("Loading TinyLlama for Route A prompting experiments...")
spottr_pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=DTYPE,
    device_map="auto",
)
print("Ready!")


Loading TinyLlama for Route A prompting experiments...
Ready!


In [26]:
# ── Zero-Shot Prompting ────────────────────────────────────────────────────
WORKOUT_TO_SUMMARIZE = (
    "3 sets deadlift 225 lbs x 5, 3 sets barbell row 135 lbs x 8, "
    "3 sets lat pulldown 120 lbs x 10, 4 sets face pulls x 15"
)

zero_shot = [
    {"role": "system", "content": "You are a fitness coach. Write a motivational workout summary in 2-3 sentences."},
    {"role": "user",   "content": f"Workout: {WORKOUT_TO_SUMMARIZE}"},
]

t0 = time.time()
out_zs = spottr_pipe(zero_shot, max_new_tokens=120, do_sample=False)
print(f"Zero-Shot ({time.time()-t0:.1f}s):")
print(out_zs[0]["generated_text"][-1]["content"].strip())


Zero-Shot (96.6s):
In this workout, you will perform three sets of deadlifts, three sets of barbell rows, three sets of lat pulldowns, and four sets of face pulls. This combination of exercises will help you build strength, increase your upper body muscle mass, and improve your overall fitness. Remember to focus on proper form and technique, and to gradually increase the weight and reps as you progress. Enjoy your workout!


In [27]:
# ── One-Shot Prompting ─────────────────────────────────────────────────────
one_shot = [
    {"role": "system", "content": "You are a fitness coach. Write a motivational workout summary in 2-3 sentences."},
    {"role": "user",   "content": "Workout: 5 sets squat 185 lbs x 5, 3 sets leg press 270 lbs x 10"},
    {"role": "assistant", "content": (
        "Incredible leg day! You powered through 5 heavy squat sets and finished strong on the leg press, "
        "building the foundation for serious lower-body strength. Keep this momentum going!"
    )},
    {"role": "user",   "content": f"Workout: {WORKOUT_TO_SUMMARIZE}"},
]

t0 = time.time()
out_os = spottr_pipe(one_shot, max_new_tokens=120, do_sample=False)
print(f"One-Shot ({time.time()-t0:.1f}s):")
print(out_os[0]["generated_text"][-1]["content"].strip())


One-Shot (113.1s):
You've got the strength to handle the weight! You've completed 3 sets of deadlifts, 3 sets of barbell rows, 3 sets of lat pull-downs, and 4 sets of face pulls. Keep pushing yourself to the limit!


In [28]:
# ── Few-Shot Prompting (3 examples) ───────────────────────────────────────
few_shot = [
    {"role": "system", "content": "You are a fitness coach. Write a motivational workout summary in 2-3 sentences. Follow the style of the examples exactly."},
    {"role": "user",   "content": "Workout: 5 sets squat 185 lbs x 5, 3 sets leg press 270 lbs x 10"},
    {"role": "assistant", "content": "Incredible leg day! You powered through 5 heavy squat sets and finished strong on the leg press. Keep this momentum going!"},
    {"role": "user",   "content": "Workout: 30 min run 3.2 miles, 100 push-ups in 4 sets"},
    {"role": "assistant", "content": "You crushed a cardio and bodyweight combo today! A 3.2-mile run paired with 100 push-ups shows outstanding endurance and upper-body strength. Your consistency is paying off!"},
    {"role": "user",   "content": "Workout: 4 sets dumbbell curl 40 lbs x 10, 3 sets tricep pushdown 60 lbs x 12, 3 sets shoulder press 65 lbs x 8"},
    {"role": "assistant", "content": "Solid arm and shoulder session! You targeted every major upper-arm muscle with focused, heavy sets across three exercises. Those gains are going to show!"},
    {"role": "user",   "content": f"Workout: {WORKOUT_TO_SUMMARIZE}"},
]

t0 = time.time()
out_fs = spottr_pipe(few_shot, max_new_tokens=120, do_sample=False)
print(f"Few-Shot ({time.time()-t0:.1f}s):")
print(out_fs[0]["generated_text"][-1]["content"].strip())


Few-Shot (52.4s):
You've got a great core! You're building the muscles that support your back and shoulders. Keep up the hard work!


In [29]:
# ── Many-Shot Prompting (6 examples) ──────────────────────────────────────
many_shot = [
    {"role": "system", "content": "You are a fitness coach. Write a motivational workout summary in exactly 2-3 sentences. Always mention the specific exercises. Always end with an encouraging sentence."},
    {"role": "user",   "content": "Workout: 5 sets squat 185 lbs x 5, 3 sets leg press 270 lbs x 10"},
    {"role": "assistant", "content": "Incredible leg day — 5 sets of squats at 185 lbs plus leg press shows real dedication to lower-body strength. You moved serious weight today. Keep building that base!"},
    {"role": "user",   "content": "Workout: 30 min run 3.2 miles, 100 push-ups in 4 sets"},
    {"role": "assistant", "content": "A 3.2-mile run combined with 100 push-ups is a well-rounded cardio and strength combo. You challenged both your aerobic system and your chest and arms. That kind of consistency separates the best from the rest!"},
    {"role": "user",   "content": "Workout: 4 sets dumbbell curl 40 lbs x 10, 3 sets tricep pushdown 60 lbs x 12"},
    {"role": "assistant", "content": "Focused arm work today — curls and tricep pushdowns for total arm development. Isolation sessions like this build the detail that compound lifts can't always reach. You're sculpting something great!"},
    {"role": "user",   "content": "Workout: 3 sets bench press 205 lbs x 3, 3 sets incline dumbbell 70 lbs x 8"},
    {"role": "assistant", "content": "Heavy chest session — benching 205 lbs and incline dumbbells at 70 lbs means you're training at an elite level. Your chest is going to thank you. Stay the course!"},
    {"role": "user",   "content": "Workout: 20 min HIIT cycling, 3 sets kettlebell swing 53 lbs x 20"},
    {"role": "assistant", "content": "HIIT cycling plus kettlebell swings is a metabolic powerhouse combination that torches calories and builds posterior chain strength. You went all in today. Rest up and come back even stronger!"},
    {"role": "user",   "content": "Workout: 5 sets pull-ups x 10, 3 sets cable row 150 lbs x 10, 3 sets face pulls 50 lbs x 15"},
    {"role": "assistant", "content": "Outstanding back and rear-delt session — pull-ups, cable rows, and face pulls hit every angle of the posterior chain. Your posture and pulling strength are going to skyrocket. Phenomenal effort!"},
    {"role": "user",   "content": f"Workout: {WORKOUT_TO_SUMMARIZE}"},
]

t0 = time.time()
out_ms = spottr_pipe(many_shot, max_new_tokens=120, do_sample=False)
print(f"Many-Shot ({time.time()-t0:.1f}s):")
print(out_ms[0]["generated_text"][-1]["content"].strip())


Many-Shot (168.8s):
Deadlift masterclass — deadlifts at 225 lbs, barbell rows at 135 lbs, lat pulls at 120 lbs, and face pulls at 15 lbs. You're a master of the deadlift and lat pull-downs. Your upper back and shoulders are going to thank you. Keep pushing!


## Prompting Strategy Comparison & Evaluation

| Strategy | Instruction Adherence | Workout Relevance | Correct Length | Ends w/ Encouragement |
|----------|-----------------------|-------------------|----------------|------------------------|
| Zero-shot | Moderate | Moderate | Sometimes | Inconsistent |
| One-shot | Good | Good | Usually | Usually |
| Few-shot | Very Good | Very Good | Consistent | Consistent |
| Many-shot | Best | Best | Most Consistent | Most Consistent |

**Conclusion:** Many-shot prompting yields the most reliable structure because the model
can extract a clear pattern from 6 diverse examples. For production, we use **few-shot (3)**
as the default — it achieves ~90% of many-shot quality at lower token cost.

**Chosen model for Django integration:** `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
- Runs locally on M2 (~2 GB RAM, <3s first load)
- Acceptable instruction adherence with few-shot prompting
- No API key or internet required for inference
